# Step 3.2 — Role-Activity Mapping from BPIC15 Logs

**Goal**: mine worker/actor assignment patterns per activity and export a reusable artifact for simulation and policy logic.

## What this step produces

1. `output/role_activity_map.json` with activity-to-role distributions (global + municipality-specific).
2. `output/role_activity_summary.csv` for quick inspection and debugging.

Role sources mined from OCEL event attributes:
- `resource`
- `Responsible_actor`
- `monitoringResource`

In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

DATASET_DIR = Path('./dataset')
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROLE_MAP_PATH = OUTPUT_DIR / 'role_activity_map.json'
ROLE_SUMMARY_CSV_PATH = OUTPUT_DIR / 'role_activity_summary.csv'

MUNICIPALITIES = [1, 2, 3, 4, 5]

print('Dataset dir:', DATASET_DIR.resolve())
print('Output dir :', OUTPUT_DIR.resolve())

Dataset dir: /home/praharsha/Desktop/bueracratic-workflow-analyzer/dataset
Output dir : /home/praharsha/Desktop/bueracratic-workflow-analyzer/output


In [2]:
def load_ocel(path: Path) -> dict:
    with open(path, 'r') as f:
        return json.load(f)

def _counter_to_distribution(counter_obj: Counter):
    if not counter_obj:
        return []
    total = float(sum(counter_obj.values()))
    if total <= 0:
        return []
    return [
        {'label': str(name), 'count': int(cnt), 'prob': float(cnt) / total}
        for name, cnt in counter_obj.most_common()
    ]

def _top_item(counter_obj: Counter):
    if not counter_obj:
        return {'label': None, 'share': 0.0, 'count': 0}
    label, count = counter_obj.most_common(1)[0]
    total = float(sum(counter_obj.values()))
    share = (float(count) / total) if total > 0 else 0.0
    return {'label': str(label), 'share': float(share), 'count': int(count)}

def _normalized_entropy(counter_obj: Counter):
    if not counter_obj:
        return 0.0
    probs = np.array(list(counter_obj.values()), dtype=float)
    total = probs.sum()
    if total <= 0:
        return 0.0
    probs = probs / total
    if len(probs) <= 1:
        return 0.0
    entropy = -np.sum(probs * np.log2(np.clip(probs, 1e-12, 1.0)))
    max_entropy = np.log2(len(probs))
    return float(entropy / max_entropy) if max_entropy > 0 else 0.0

def mine_role_activity_map(logs_by_municipality):
    by_m = defaultdict(lambda: defaultdict(lambda: {
        'resource': Counter(),
        'Responsible_actor': Counter(),
        'monitoringResource': Counter(),
        'n_events': 0,
    }))

    global_map = defaultdict(lambda: {
        'resource': Counter(),
        'Responsible_actor': Counter(),
        'monitoringResource': Counter(),
        'n_events': 0,
    })

    for m, log in logs_by_municipality.items():
        events = log.get('ocel:events', {})
        for ev in events.values():
            activity = str(ev.get('ocel:activity', 'UNKNOWN_ACTIVITY'))
            vmap = ev.get('ocel:vmap', {}) or {}

            resource = str(vmap.get('resource', '')).strip()
            actor = str(vmap.get('Responsible_actor', '')).strip()
            monitoring_resource = str(vmap.get('monitoringResource', '')).strip()

            local_bucket = by_m[int(m)][activity]
            global_bucket = global_map[activity]

            local_bucket['n_events'] += 1
            global_bucket['n_events'] += 1

            if resource:
                local_bucket['resource'][resource] += 1
                global_bucket['resource'][resource] += 1
            if actor:
                local_bucket['Responsible_actor'][actor] += 1
                global_bucket['Responsible_actor'][actor] += 1
            if monitoring_resource:
                local_bucket['monitoringResource'][monitoring_resource] += 1
                global_bucket['monitoringResource'][monitoring_resource] += 1

    payload = {
        'summary': {},
        'by_municipality': {},
        'global_by_activity': {},
    }

    flat_rows = []

    for m, activity_map in by_m.items():
        m_key = str(int(m))
        payload['by_municipality'][m_key] = {}

        for activity, bucket in activity_map.items():
            top_res = _top_item(bucket['resource'])
            top_actor = _top_item(bucket['Responsible_actor'])

            row_payload = {
                'n_events': int(bucket['n_events']),
                'resource_candidates': int(len(bucket['resource'])),
                'actor_candidates': int(len(bucket['Responsible_actor'])),
                'monitoring_candidates': int(len(bucket['monitoringResource'])),
                'top_resource': top_res['label'],
                'top_resource_share': float(top_res['share']),
                'top_actor': top_actor['label'],
                'top_actor_share': float(top_actor['share']),
                'resource_entropy_norm': _normalized_entropy(bucket['resource']),
                'actor_entropy_norm': _normalized_entropy(bucket['Responsible_actor']),
                'resource_distribution': _counter_to_distribution(bucket['resource']),
                'actor_distribution': _counter_to_distribution(bucket['Responsible_actor']),
                'monitoring_distribution': _counter_to_distribution(bucket['monitoringResource']),
            }

            payload['by_municipality'][m_key][activity] = row_payload

            flat_rows.append({
                'scope': 'municipality',
                'municipality': int(m),
                'activity': activity,
                'n_events': int(bucket['n_events']),
                'resource_candidates': int(len(bucket['resource'])),
                'actor_candidates': int(len(bucket['Responsible_actor'])),
                'top_resource': top_res['label'],
                'top_resource_share': float(top_res['share']),
                'top_actor': top_actor['label'],
                'top_actor_share': float(top_actor['share']),
                'resource_entropy_norm': _normalized_entropy(bucket['resource']),
                'actor_entropy_norm': _normalized_entropy(bucket['Responsible_actor']),
            })

    for activity, bucket in global_map.items():
        top_res = _top_item(bucket['resource'])
        top_actor = _top_item(bucket['Responsible_actor'])

        payload['global_by_activity'][activity] = {
            'n_events': int(bucket['n_events']),
            'resource_candidates': int(len(bucket['resource'])),
            'actor_candidates': int(len(bucket['Responsible_actor'])),
            'monitoring_candidates': int(len(bucket['monitoringResource'])),
            'top_resource': top_res['label'],
            'top_resource_share': float(top_res['share']),
            'top_actor': top_actor['label'],
            'top_actor_share': float(top_actor['share']),
            'resource_entropy_norm': _normalized_entropy(bucket['resource']),
            'actor_entropy_norm': _normalized_entropy(bucket['Responsible_actor']),
            'resource_distribution': _counter_to_distribution(bucket['resource']),
            'actor_distribution': _counter_to_distribution(bucket['Responsible_actor']),
            'monitoring_distribution': _counter_to_distribution(bucket['monitoringResource']),
        }

        flat_rows.append({
            'scope': 'global',
            'municipality': None,
            'activity': activity,
            'n_events': int(bucket['n_events']),
            'resource_candidates': int(len(bucket['resource'])),
            'actor_candidates': int(len(bucket['Responsible_actor'])),
            'top_resource': top_res['label'],
            'top_resource_share': float(top_res['share']),
            'top_actor': top_actor['label'],
            'top_actor_share': float(top_actor['share']),
            'resource_entropy_norm': _normalized_entropy(bucket['resource']),
            'actor_entropy_norm': _normalized_entropy(bucket['Responsible_actor']),
        })

    payload['summary'] = {
        'description': 'Role-activity mapping mined from BPIC15 OCEL logs',
        'municipalities': sorted([int(m) for m in by_m.keys()]),
        'n_municipality_activity_rows': int(sum(len(v) for v in payload['by_municipality'].values())),
        'n_global_activities': int(len(payload['global_by_activity'])),
    }

    flat_df = pd.DataFrame(flat_rows)
    return payload, flat_df


logs = {}
for m in MUNICIPALITIES:
    path = DATASET_DIR / f'BPIC15_Municipality{m}.jsonocel'
    if not path.exists():
        print(f'Warning: missing {path.name}')
        continue
    logs[m] = load_ocel(path)

print('Loaded municipalities:', sorted(logs.keys()))

Loaded municipalities: [1, 2, 3, 4, 5]


In [3]:
role_activity_map, role_summary_df = mine_role_activity_map(logs)

with open(ROLE_MAP_PATH, 'w') as f:
    json.dump(role_activity_map, f, indent=2)

role_summary_df.to_csv(ROLE_SUMMARY_CSV_PATH, index=False)

print('Saved role map JSON :', ROLE_MAP_PATH.resolve())
print('Saved summary CSV   :', ROLE_SUMMARY_CSV_PATH.resolve())
print('Rows in summary CSV :', len(role_summary_df))
print('Global activities   :', role_activity_map['summary']['n_global_activities'])
print('Muni-activity rows  :', role_activity_map['summary']['n_municipality_activity_rows'])

Saved role map JSON : /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/role_activity_map.json
Saved summary CSV   : /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/role_activity_summary.csv
Rows in summary CSV : 1783
Global activities   : 356
Muni-activity rows  : 1427


In [4]:
# Quick sanity view: activities with strongest actor specialization
global_df = role_summary_df[role_summary_df['scope'] == 'global'].copy()
global_df = global_df.sort_values(['top_actor_share', 'n_events'], ascending=[False, False])

display_cols = [
    'activity', 'n_events',
    'actor_candidates', 'top_actor', 'top_actor_share', 'actor_entropy_norm',
    'resource_candidates', 'top_resource', 'top_resource_share', 'resource_entropy_norm'
]

print('Top 20 globally specialized activities (by actor share):')
display(global_df[display_cols].head(20))

print('Municipality-level sample (M1 top 20 by actor share):')
m1 = role_summary_df[(role_summary_df['scope'] == 'municipality') & (role_summary_df['municipality'] == 1)]
m1 = m1.sort_values(['top_actor_share', 'n_events'], ascending=[False, False])
display(m1[display_cols].head(20))

Top 20 globally specialized activities (by actor share):


,activity,n_events,actor_candidates,top_actor,top_actor_share,actor_entropy_norm,resource_candidates,top_resource,top_resource_share,resource_entropy_norm
1752,enter enddate suspension art. 35 WABO,2,1,560598,1.0,0.0,2,560532.0,0.5,1.0
1754,appeal system,2,1,560458,1.0,0.0,1,560458.0,1.0,0.0
1755,phase appeal to higher court,2,1,560458,1.0,0.0,1,560458.0,1.0,0.0
1756,treating appeal to higher court subcase,2,1,560458,1.0,0.0,1,560458.0,1.0,0.0
1782,handle perspective in system,2,1,560604,1.0,0.0,2,560604.0,0.5,1.0
1641,treat subcases submit draft decision,1,1,560464,1.0,0.0,1,2670601.0,1.0,0.0
1642,subcases submit draft decision completed,1,1,560464,1.0,0.0,1,2670601.0,1.0,0.0
1678,send letter receptive test not ok wait for rep...,1,1,560464,1.0,0.0,1,2670601.0,1.0,0.0
1687,reactive designation received,1,1,4901428,1.0,0.0,1,560462.0,1.0,0.0
1688,subcase designation GS completed,1,1,4901428,1.0,0.0,1,560462.0,1.0,0.0


Municipality-level sample (M1 top 20 by actor share):


,activity,n_events,actor_candidates,top_actor,top_actor_share,actor_entropy_norm,resource_candidates,top_resource,top_resource_share,resource_entropy_norm
225,term to perspective,5,1,4901428,1.0,0.0,2,560872.0,0.8,0.721928
270,enter senddate letter no permit required,2,1,4901428,1.0,0.0,1,560872.0,1.0,0.000000
116,temporary permit,1,1,560464,1.0,0.0,1,560462.0,1.0,0.000000
167,request updated plan again,1,1,4901428,1.0,0.0,1,2670601.0,1.0,0.000000
212,enrich draft decision environmental permit,1,1,4901428,1.0,0.0,1,560890.0,1.0,0.000000
214,treat subcases submit draft decision,1,1,560464,1.0,0.0,1,2670601.0,1.0,0.000000
215,subcases submit draft decision completed,1,1,560464,1.0,0.0,1,2670601.0,1.0,0.000000
229,date environmental permit irrevocable,1,1,4901428,1.0,0.0,1,3273854.0,1.0,0.000000
232,send send confirmation competent authority,1,1,560950,1.0,0.0,1,560872.0,1.0,0.000000
233,create forwarding copy,1,1,560950,1.0,0.0,1,560872.0,1.0,0.000000


## Step 3.2 complete

You now have a reusable role-mapping artifact for downstream simulator/training integration:
- `output/role_activity_map.json`
- `output/role_activity_summary.csv`